# 03 — ETL et chargement dans le Data Warehouse
### CHU Mohammed VI — Agadir | Module Hospitalisations

**Objectif de ce notebook :**
1. **Extract** : lire les CSV nettoyés (`data/clean/`)
2. **Transform** : standardiser les formats, calculer `duree_sejour` et `tranche_age`, harmoniser les libellés de services
3. **Load — staging** : charger les données transformées dans des tables intermédiaires MySQL
4. **Load — DWH** : construire et peupler le schéma étoile (`DimPatient`, `DimTemps`, `DimService`, `DimMedecin`, `FactHospitalisation`)

Prérequis : le fichier `.env` à la racine du projet doit contenir tes identifiants MySQL, et `sql/schema_dwh.sql` doit avoir été exécuté une fois sur ta base `chu_dwh` (ou sera exécuté automatiquement par ce notebook, voir section 4).

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

CLEAN_DIR = "../data/clean"

load_dotenv("../.env")

print("Librairies chargées avec succès ")

Librairies chargées avec succès 


## 1. Connexion à MySQL (via .env, aucun mot de passe en dur)

In [2]:
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "chu_dwh")
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD")

if DB_PASSWORD is None:
    raise ValueError("DB_PASSWORD introuvable : vérifie que le fichier .env existe bien à la racine du projet.")

connection_string = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Test de connexion
with engine.connect() as conn:
    resultat = conn.execute(text("SELECT DATABASE();"))
    print("Connecté avec succès à la base :", resultat.fetchone()[0])

Connecté avec succès à la base : chu_dwh


## 2. Extract — lecture des CSV nettoyés

In [3]:
df_patients = pd.read_csv(f"{CLEAN_DIR}/patients.csv", parse_dates=["date_naissance"])
df_services = pd.read_csv(f"{CLEAN_DIR}/services.csv")
df_medecins = pd.read_csv(f"{CLEAN_DIR}/medecins.csv")
df_hospitalisations = pd.read_csv(f"{CLEAN_DIR}/hospitalisations.csv", parse_dates=["date_entree", "date_sortie"])

print(f"Patients : {len(df_patients)} | Services : {len(df_services)} | Médecins : {len(df_medecins)} | Hospitalisations : {len(df_hospitalisations)}")

Patients : 8000 | Services : 16 | Médecins : 103 | Hospitalisations : 14580


## 3. Transform

### 3.1 Standardisation des formats

In [4]:
# --- Standardisation des chaînes de caractères (espaces, casse) ---
for col in ["nom", "prenom", "ville"]:
    df_patients[col] = df_patients[col].astype(str).str.strip().str.title()

df_services["nom_service"] = df_services["nom_service"].astype(str).str.strip().str.title()
df_services["categorie"] = df_services["categorie"].astype(str).str.strip().str.title()

for col in ["nom", "prenom", "specialite"]:
    df_medecins[col] = df_medecins[col].astype(str).str.strip().str.title()

df_hospitalisations["type_admission"] = df_hospitalisations["type_admission"].astype(str).str.strip().str.capitalize()
df_hospitalisations["pathologie"] = df_hospitalisations["pathologie"].astype(str).str.strip()
df_hospitalisations["mode_sortie"] = df_hospitalisations["mode_sortie"].astype(str).str.strip()

print("Standardisation des formats appliquée")

Standardisation des formats appliquée


In [5]:
# --- Harmonisation des libellés de services ---
# Utile si des variantes d'écriture existent (ex. "chirurgie generale" vs "Chirurgie Générale").
# On construit un dictionnaire de correspondance à partir des valeurs uniques observées,
# pour repérer d'éventuelles variantes orthographiques à corriger manuellement si besoin.

libelles_uniques = df_services["nom_service"].unique()
print("Libellés de services après standardisation :")
for lib in sorted(libelles_uniques):
    print(" -", lib)

# Exemple de dictionnaire de correction si des variantes apparaissaient (à adapter si besoin) :
CORRECTIONS_LIBELLES = {
    # "Chirurgie General": "Chirurgie Générale",
}
df_services["nom_service"] = df_services["nom_service"].replace(CORRECTIONS_LIBELLES)

Libellés de services après standardisation :
 - Cardiologie
 - Chirurgie Générale
 - Dermatologie
 - Gastro-Entérologie
 - Maternité
 - Médecine Interne
 - Neurologie
 - Néphrologie
 - Oncologie
 - Orl
 - Orthopédie
 - Pneumologie
 - Pédiatrie
 - Réanimation
 - Urgences
 - Urologie


### 3.2 Calcul des indicateurs dérivés

In [6]:
# --- duree_sejour ---
df_hospitalisations["duree_sejour"] = (
    df_hospitalisations["date_sortie"] - df_hospitalisations["date_entree"]
).dt.days

# Sécurité : aucune durée ne doit être négative après le nettoyage du notebook 02
assert (df_hospitalisations["duree_sejour"] >= 0).all(), "Durées de séjour négatives détectées !"

print(df_hospitalisations["duree_sejour"].describe())

count    14580.000000
mean         9.237449
std          4.089991
min          2.000000
25%          6.000000
50%          9.000000
75%         12.000000
max         29.000000
Name: duree_sejour, dtype: float64


In [7]:
# --- tranche_age ---
def calculer_tranche_age(age):
    if pd.isna(age):
        return "Non renseigné"
    if age <= 18:
        return "0-18"
    elif age <= 40:
        return "19-40"
    elif age <= 60:
        return "41-60"
    else:
        return "60+"

df_patients["tranche_age"] = df_patients["age"].apply(calculer_tranche_age)

df_patients["tranche_age"].value_counts()

tranche_age
60+      2793
41-60    1887
19-40    1725
0-18     1595
Name: count, dtype: int64

### 3.3 Aperçu des données transformées

In [8]:
df_hospitalisations[["id_hospitalisation", "date_entree", "date_sortie", "duree_sejour"]].head()

,id_hospitalisation,date_entree,date_sortie,duree_sejour
0,1,2024-06-26,2024-07-06,10
1,2,2024-03-03,2024-03-09,6
2,3,2023-12-18,2024-01-08,21
3,4,2024-01-10,2024-01-16,6
4,6,2026-06-06,2026-06-11,5


## 4. Load — tables de staging

Chargement brut des données transformées dans des tables intermédiaires (`staging_*`), avant modélisation en schéma étoile. Utile pour garder une trace intermédiaire et déboguer facilement si besoin.

In [9]:
df_patients.to_sql("staging_patients", engine, if_exists="replace", index=False)
df_services.to_sql("staging_services", engine, if_exists="replace", index=False)
df_medecins.to_sql("staging_medecins", engine, if_exists="replace", index=False)
df_hospitalisations.to_sql("staging_hospitalisations", engine, if_exists="replace", index=False)

print("Tables de staging chargées avec succès")

Tables de staging chargées avec succès


## 5. Création du schéma étoile (DWH)

Exécute le script `sql/schema_dwh.sql` directement depuis le notebook (idempotent : les tables sont recréées à chaque exécution via `DROP TABLE IF EXISTS`).

In [10]:
with open("../sql/schema_dwh.sql", "r", encoding="utf-8") as f:
    script_sql = f.read()

# Découpage sur les points-virgules pour exécuter chaque instruction séparément
instructions = [instr.strip() for instr in script_sql.split(";") if instr.strip()]

with engine.begin() as conn:
    for instruction in instructions:
        conn.execute(text(instruction))

print(f"{len(instructions)} instructions SQL exécutées — schéma étoile créé")

10 instructions SQL exécutées — schéma étoile créé


## 6. Peuplement de `DimService`

In [11]:
dim_service = df_services[["id_service", "nom_service", "categorie", "capacite_lits", "etage"]].copy()
dim_service.to_sql("DimService", engine, if_exists="append", index=False)

print(f"DimService peuplée : {len(dim_service)} lignes")

DimService peuplée : 16 lignes


C:\Users\0oo-u\AppData\Local\Temp\ipykernel_2500\2490530937.py:2: UserWarning: The provided table name 'DimService' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  dim_service.to_sql("DimService", engine, if_exists="append", index=False)


## 7. Peuplement de `DimMedecin`

In [12]:
dim_medecin = df_medecins[["id_medecin", "nom", "prenom", "specialite", "id_service", "annees_experience"]].copy()
dim_medecin.to_sql("DimMedecin", engine, if_exists="append", index=False)

print(f"DimMedecin peuplée : {len(dim_medecin)} lignes")

DimMedecin peuplée : 103 lignes


C:\Users\0oo-u\AppData\Local\Temp\ipykernel_2500\1914107353.py:2: UserWarning: The provided table name 'DimMedecin' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  dim_medecin.to_sql("DimMedecin", engine, if_exists="append", index=False)


## 8. Peuplement de `DimPatient`

In [13]:
dim_patient = df_patients[["id_patient", "age", "tranche_age", "sexe", "ville", "groupe_sanguin", "couverture_sante"]].copy()
dim_patient["age"] = dim_patient["age"].round().astype(int)
dim_patient.to_sql("DimPatient", engine, if_exists="append", index=False)

print(f"DimPatient peuplée : {len(dim_patient)} lignes")

DimPatient peuplée : 8000 lignes


C:\Users\0oo-u\AppData\Local\Temp\ipykernel_2500\1887765042.py:3: UserWarning: The provided table name 'DimPatient' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  dim_patient.to_sql("DimPatient", engine, if_exists="append", index=False)


## 9. Peuplement de `DimTemps`

Génère une ligne par jour calendaire couvrant toute la plage de dates du dataset (des `date_entree` les plus anciennes aux `date_sortie` les plus récentes).

In [14]:
date_min = df_hospitalisations["date_entree"].min()
date_max = df_hospitalisations["date_sortie"].max()

toutes_dates = pd.date_range(start=date_min, end=date_max, freq="D")

JOURS_FR = {
    "Monday": "Lundi", "Tuesday": "Mardi", "Wednesday": "Mercredi", "Thursday": "Jeudi",
    "Friday": "Vendredi", "Saturday": "Samedi", "Sunday": "Dimanche",
}

dim_temps = pd.DataFrame({"date_complete": toutes_dates})
dim_temps["id_temps"] = dim_temps["date_complete"].dt.strftime("%Y%m%d").astype(int)
dim_temps["jour"] = dim_temps["date_complete"].dt.day
dim_temps["mois"] = dim_temps["date_complete"].dt.month
dim_temps["trimestre"] = dim_temps["date_complete"].dt.quarter
dim_temps["annee"] = dim_temps["date_complete"].dt.year
dim_temps["jour_semaine"] = dim_temps["date_complete"].dt.day_name().map(JOURS_FR)

dim_temps = dim_temps[["id_temps", "date_complete", "jour", "mois", "trimestre", "annee", "jour_semaine"]]
dim_temps.to_sql("DimTemps", engine, if_exists="append", index=False)

print(f"DimTemps peuplée : {len(dim_temps)} lignes (du {date_min.date()} au {date_max.date()})")

DimTemps peuplée : 1115 lignes (du 2023-07-22 au 2026-08-09)


C:\Users\0oo-u\AppData\Local\Temp\ipykernel_2500\1896716692.py:20: UserWarning: The provided table name 'DimTemps' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  dim_temps.to_sql("DimTemps", engine, if_exists="append", index=False)


## 10. Peuplement de `FactHospitalisation`

La clé `id_temps` de la table de faits est résolue à partir de `date_entree` (date d'ancrage retenue pour l'hospitalisation).

In [15]:
fact_hospitalisation = df_hospitalisations.copy()
fact_hospitalisation["id_temps"] = fact_hospitalisation["date_entree"].dt.strftime("%Y%m%d").astype(int)
fact_hospitalisation["jours_occupation"] = fact_hospitalisation["duree_sejour"]

# id_medecin peut valoir -1 (non renseigné) : MySQL refusera la FK si -1 n'existe pas dans DimMedecin.
# On le convertit donc en NULL pour respecter l'intégrité référentielle.
fact_hospitalisation["id_medecin"] = fact_hospitalisation["id_medecin"].replace(-1, np.nan)

colonnes_fait = [
    "id_patient", "id_temps", "id_service", "id_medecin",
    "duree_sejour", "jours_occupation", "type_admission", "pathologie", "mode_sortie",
]
fact_hospitalisation = fact_hospitalisation[colonnes_fait]

fact_hospitalisation.to_sql("FactHospitalisation", engine, if_exists="append", index=False)

print(f"FactHospitalisation peuplée : {len(fact_hospitalisation)} lignes")

FactHospitalisation peuplée : 14580 lignes


C:\Users\0oo-u\AppData\Local\Temp\ipykernel_2500\1962981613.py:15: UserWarning: The provided table name 'FactHospitalisation' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  fact_hospitalisation.to_sql("FactHospitalisation", engine, if_exists="append", index=False)


## 11. Vérification finale — comptage des lignes dans chaque table du DWH

In [16]:
tables_dwh = ["DimPatient", "DimTemps", "DimService", "DimMedecin", "FactHospitalisation"]

for table in tables_dwh:
    count = pd.read_sql(f"SELECT COUNT(*) AS nb_lignes FROM {table}", engine)
    print(f"{table} : {count['nb_lignes'].iloc[0]} lignes")

DimPatient : 8000 lignes
DimTemps : 1115 lignes
DimService : 16 lignes
DimMedecin : 103 lignes
FactHospitalisation : 14580 lignes


In [17]:
# Aperçu d'une jointure complète pour valider le schéma étoile
requete_test = """
SELECT
    f.id_fait,
    p.ville,
    p.tranche_age,
    s.nom_service,
    m.nom AS nom_medecin,
    t.annee,
    t.mois,
    f.duree_sejour,
    f.pathologie
FROM FactHospitalisation f
JOIN DimPatient p ON f.id_patient = p.id_patient
JOIN DimService s ON f.id_service = s.id_service
JOIN DimTemps t ON f.id_temps = t.id_temps
LEFT JOIN DimMedecin m ON f.id_medecin = m.id_medecin
LIMIT 10;
"""

pd.read_sql(requete_test, engine)

,id_fait,ville,tranche_age,nom_service,nom_medecin,annee,mois,duree_sejour,pathologie
0,1,Sidi Ifni,0-18,Cardiologie,Dahbi,2024,6,10,Insuffisance cardiaque
1,20,Ait Melloul,0-18,Cardiologie,Cherkaoui,2026,6,16,Infarctus du myocarde
2,25,Essaouira,19-40,Cardiologie,El Ouazzani,2023,11,14,Infarctus du myocarde
3,33,Tanger,0-18,Cardiologie,Lahlou,2025,8,9,Insuffisance cardiaque
4,46,Chtouka Aït Baha,60+,Cardiologie,Benjelloun,2025,2,12,Infarctus du myocarde
5,64,Tétouan,41-60,Cardiologie,Benjelloun,2024,5,3,Hypertension artérielle sévère
6,84,Kénitra,41-60,Cardiologie,El Fenni,2025,7,9,Hypertension artérielle sévère
7,95,Safi,60+,Cardiologie,Benjelloun,2026,1,8,Angine de poitrine
8,128,Rabat,19-40,Cardiologie,El Fassi,2025,7,5,Hypertension artérielle sévère
9,134,Tanger,41-60,Cardiologie,Filali,2026,4,11,Insuffisance cardiaque


In [18]:
# --- Diagnostic : pourquoi certaines hospitalisations n'ont pas de service valide ---

requete_null = "SELECT COUNT(*) AS nb_id_service_null FROM FactHospitalisation WHERE id_service IS NULL;"
resultat_null = pd.read_sql(requete_null, engine)
print("Lignes avec id_service NULL :")
print(resultat_null)

requete_invalide = """
SELECT COUNT(*) AS nb_id_service_invalide
FROM FactHospitalisation f
LEFT JOIN DimService s ON f.id_service = s.id_service
WHERE f.id_service IS NOT NULL AND s.id_service IS NULL;
"""
resultat_invalide = pd.read_sql(requete_invalide, engine)
print("\nLignes avec id_service invalide (n'existe pas dans DimService) :")
print(resultat_invalide)

Lignes avec id_service NULL :
   nb_id_service_null
0                   0

Lignes avec id_service invalide (n'existe pas dans DimService) :
   nb_id_service_invalide
0                       0


In [19]:
# --- Re-diagnostic complet, à l'instant T ---

print("Contenu de DimService :")
print(pd.read_sql("SELECT id_service, nom_service FROM DimService ORDER BY id_service;", engine))

print("\nNombre total de lignes dans DimService :")
print(pd.read_sql("SELECT COUNT(*) AS nb FROM DimService;", engine))

print("\nid_service NULL dans FactHospitalisation :")
print(pd.read_sql("SELECT COUNT(*) AS nb FROM FactHospitalisation WHERE id_service IS NULL;", engine))

print("\nid_service invalide (pas dans DimService) :")
print(pd.read_sql("""
    SELECT COUNT(*) AS nb
    FROM FactHospitalisation f
    LEFT JOIN DimService s ON f.id_service = s.id_service
    WHERE f.id_service IS NOT NULL AND s.id_service IS NULL;
""", engine))

print("\nServices avec nom_service NULL ou vide dans DimService :")
print(pd.read_sql("SELECT * FROM DimService WHERE nom_service IS NULL OR TRIM(nom_service) = '';", engine))

Contenu de DimService :
    id_service         nom_service
0            1         Cardiologie
1            2           Pédiatrie
2            3  Chirurgie Générale
3            4            Urgences
4            5           Maternité
5            6           Oncologie
6            7         Néphrologie
7            8    Médecine Interne
8            9          Orthopédie
9           10         Pneumologie
10          11  Gastro-Entérologie
11          12            Urologie
12          13          Neurologie
13          14         Réanimation
14          15        Dermatologie
15          16                 Orl

Nombre total de lignes dans DimService :
   nb
0  16

id_service NULL dans FactHospitalisation :
   nb
0   0

id_service invalide (pas dans DimService) :
   nb
0   0

Services avec nom_service NULL ou vide dans DimService :
Empty DataFrame
Columns: [id_service, nom_service, categorie, capacite_lits, etage]
Index: []


---
### Prochaine étape
Le Data Warehouse est peuplé et interrogeable. Ouvrir `04_analyse_exploratoire_kpi.ipynb` pour l'analyse exploratoire (EDA) et le calcul des KPI consolidés.